# 03 — Activation patching

A atribuição direta mostrou quais componentes apontam para a resposta correta. Agora faremos uma intervenção causal:

1. executamos um prompt **limpo** e guardamos suas ativações;
2. executamos a versão **corrompida**;
3. substituímos uma ativação corrompida pela ativação limpa correspondente;
4. medimos quanto do comportamento correto foi recuperado.

Os pesos do modelo não mudam. O patch é uma substituição temporária durante uma execução.

In [1]:
from functools import partial

import plotly.graph_objects as go
import torch
from transformer_lens import patching
import transformer_lens.utilities as utils

from ioi_replication import (
    build_ioi_examples,
    load_gpt2_small,
    logits_to_logit_diff,
    normalized_recovery,
    select_device,
    tokenize_answers,
)

device = select_device()
model = load_gpt2_small(device)
prompts, answers = build_ioi_examples()
tokens = model.to_tokens(prompts, prepend_bos=True).to(device)
answer_tokens = tokenize_answers(model, answers, device)

swap_indices = torch.tensor(
    [i + 1 if i % 2 == 0 else i - 1 for i in range(len(tokens))],
    device=device,
)
corrupted_tokens = tokens[swap_indices]
print(f"Dispositivo: {device}")

C:\Users\Marcos\Documents\IOI Replication\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   8%|▊         | 12/148 [00:00<00:01, 106.38it/s]

Loading weights:  17%|█▋        | 25/148 [00:00<00:01, 116.02it/s]

Loading weights:  25%|██▌       | 37/148 [00:00<00:02, 49.44it/s] 

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 230.73it/s]

Dispositivo: cuda


In [2]:
for clean_index, corrupted_index in enumerate(swap_indices.tolist()):
    print(f"limpo {clean_index}:      {prompts[clean_index]}")
    print(f"corrompido {clean_index}: {prompts[corrupted_index]}\n")

with torch.no_grad():
    clean_logits, clean_cache = model.run_with_cache(tokens)
    corrupted_logits = model(corrupted_tokens)

clean_diff = logits_to_logit_diff(clean_logits, answer_tokens)
corrupted_diff = logits_to_logit_diff(corrupted_logits, answer_tokens)
print(f"logit diff limpo:      {clean_diff.item():+.3f}")
print(f"logit diff corrompido: {corrupted_diff.item():+.3f}")

limpo 0:      When John and Mary went to the shops, John gave the bag to
corrompido 0: When John and Mary went to the shops, Mary gave the bag to

limpo 1:      When John and Mary went to the shops, Mary gave the bag to
corrompido 1: When John and Mary went to the shops, John gave the bag to

limpo 2:      When Tom and James went to the park, James gave the ball to
corrompido 2: When Tom and James went to the park, Tom gave the ball to

limpo 3:      When Tom and James went to the park, Tom gave the ball to
corrompido 3: When Tom and James went to the park, James gave the ball to

limpo 4:      When Dan and Sid went to the shops, Sid gave an apple to
corrompido 4: When Dan and Sid went to the shops, Dan gave an apple to

limpo 5:      When Dan and Sid went to the shops, Dan gave an apple to
corrompido 5: When Dan and Sid went to the shops, Sid gave an apple to

limpo 6:      After Martin and Amy went to the park, Amy gave a drink to
corrompido 6: After Martin and Amy went to the park, 

logit diff limpo:      +3.552
logit diff corrompido: -3.552


## Recuperação normalizada

Mediremos o resultado do patch em uma escala comum:

$$\text{recuperação}=\frac{\text{patched}-\text{corrompido}}{\text{limpo}-\text{corrompido}}$$

- `0`: o patch não melhorou o comportamento corrompido;
- `1`: recuperou o nível do comportamento limpo;
- valores fora de `[0, 1]` são possíveis: a intervenção pode piorar ou superar o baseline limpo.

In [3]:
def ioi_metric(logits: torch.Tensor) -> torch.Tensor:
    patched_diff = logits_to_logit_diff(logits, answer_tokens)
    return normalized_recovery(patched_diff, clean_diff, corrupted_diff)


torch.testing.assert_close(ioi_metric(clean_logits), torch.tensor(1.0, device=device))
torch.testing.assert_close(ioi_metric(corrupted_logits), torch.tensor(0.0, device=device))
midpoint_logits = (clean_logits + corrupted_logits) / 2
torch.testing.assert_close(ioi_metric(midpoint_logits), torch.tensor(0.5, device=device))
print("Endpoints verificados: corrompido=0, meio=0.5, limpo=1")

Endpoints verificados: corrompido=0, meio=0.5, limpo=1


## 1. Patching do residual stream

Para cada combinação `(camada, posição)`, substituímos somente o `resid_pre` corrompido pela versão limpa.

O eixo vertical será a camada. O eixo horizontal será a posição do token. Cores mais altas indicam maior recuperação causal.

In [4]:
with torch.no_grad():
    builtin_residual_results = patching.get_act_patch_resid_pre(
        model,
        corrupted_tokens,
        clean_cache,
        ioi_metric,
    )

assert builtin_residual_results.shape == (model.cfg.n_layers, tokens.shape[1])
position_labels = [
    f"{index}:{token!r}"
    for index, token in enumerate(model.to_str_tokens(tokens[0].detach().cpu()))
]

figure = go.Figure(
    data=go.Heatmap(
        z=builtin_residual_results.detach().cpu().numpy(),
        x=position_labels,
        y=[f"L{layer}" for layer in range(model.cfg.n_layers)],
        colorscale="RdBu",
        zmid=0,
        colorbar_title="recuperação",
        hovertemplate="camada=%{y}<br>posição=%{x}<br>recuperação=%{z:.3f}<extra></extra>",
    )
)
figure.update_layout(
    title="Patching de resid_pre por camada e posição",
    xaxis_title="posição do token no primeiro prompt",
    yaxis_title="camada",
    height=600,
)
figure.show()

best_flat = builtin_residual_results.argmax().item()
best_layer, best_position = divmod(best_flat, tokens.shape[1])
print(
    f"Maior recuperação: camada {best_layer}, posição {best_position}, "
    f"valor={builtin_residual_results.max().item():.3f}"
)

  0%|          | 0/180 [00:00<?, ?it/s]

  1%|          | 2/180 [00:00<00:11, 14.84it/s]

  2%|▏         | 4/180 [00:00<00:12, 14.15it/s]

  3%|▎         | 6/180 [00:00<00:11, 15.41it/s]

  4%|▍         | 8/180 [00:00<00:10, 16.47it/s]

  6%|▌         | 10/180 [00:00<00:09, 17.53it/s]

... (82 outputs left out)

 99%|█████████▉| 178/180 [00:10<00:00, 17.83it/s]

100%|██████████| 180/180 [00:10<00:00, 18.28it/s]

100%|██████████| 180/180 [00:10<00:00, 17.63it/s]

Maior recuperação: camada 10, posição 14, valor=1.005


## 2. Implementação manual com hooks

A função pronta acima será nosso oráculo. Agora reproduziremos o mesmo experimento explicitamente:

- o hook intercepta `resid_pre`;
- troca apenas uma posição pela ativação do cache limpo;
- deixa o restante da execução corrompida inalterado.

Se entendemos a intervenção, os dois mapas devem coincidir.

In [5]:
def patch_residual_position(
    corrupted_residual: torch.Tensor,
    hook,
    *,
    position: int,
) -> torch.Tensor:
    corrupted_residual[:, position, :] = clean_cache[hook.name][:, position, :]
    return corrupted_residual


own_residual_results = torch.zeros_like(builtin_residual_results)
with torch.no_grad():
    for layer in range(model.cfg.n_layers):
        hook_name = utils.get_act_name("resid_pre", layer)
        for position in range(tokens.shape[1]):
            patched_logits = model.run_with_hooks(
                corrupted_tokens,
                fwd_hooks=[
                    (
                        hook_name,
                        partial(
                            patch_residual_position,
                            position=position,
                        ),
                    )
                ],
            )
            own_residual_results[layer, position] = ioi_metric(patched_logits)

torch.testing.assert_close(
    own_residual_results,
    builtin_residual_results,
    rtol=1e-4,
    atol=1e-4,
)
max_error = (own_residual_results - builtin_residual_results).abs().max()
print(f"Implementação manual coincide com a função pronta; erro máximo={max_error.item():.2e}")

Implementação manual coincide com a função pronta; erro máximo=0.00e+00


## 3. Patching de heads individuais

Agora fazemos o patch da saída `z` de uma head, em todas as posições, mantendo as demais heads corrompidas.

O mapa `(camada, head)` mede recuperação causal sob esta intervenção. Isso é diferente de DLA: uma head pode transportar informação importante sem escrever diretamente nos logits.

In [6]:
with torch.no_grad():
    head_results = patching.get_act_patch_attn_head_out_all_pos(
        model,
        corrupted_tokens,
        clean_cache,
        ioi_metric,
    )

assert head_results.shape == (model.cfg.n_layers, model.cfg.n_heads)

figure = go.Figure(
    data=go.Heatmap(
        z=head_results.detach().cpu().numpy(),
        x=[f"H{head}" for head in range(model.cfg.n_heads)],
        y=[f"L{layer}" for layer in range(model.cfg.n_layers)],
        colorscale="RdBu",
        zmid=0,
        colorbar_title="recuperação",
        hovertemplate="camada=%{y}<br>head=%{x}<br>recuperação=%{z:.3f}<extra></extra>",
    )
)
figure.update_layout(
    title="Recuperação ao aplicar patch em cada head",
    xaxis_title="head",
    yaxis_title="camada",
    height=600,
)
figure.show()

best_flat = head_results.argmax().item()
best_layer, best_head = divmod(best_flat, model.cfg.n_heads)
print(
    f"Maior recuperação individual: head {best_layer}.{best_head}, "
    f"valor={head_results.max().item():.3f}"
)

  0%|          | 0/144 [00:00<?, ?it/s]

  1%|▏         | 2/144 [00:00<00:07, 19.57it/s]

  3%|▎         | 4/144 [00:00<00:07, 19.65it/s]

  4%|▍         | 6/144 [00:00<00:07, 19.27it/s]

  6%|▌         | 8/144 [00:00<00:09, 14.13it/s]

  7%|▋         | 10/144 [00:00<00:09, 14.77it/s]

... (63 outputs left out)

 98%|█████████▊| 141/144 [00:08<00:00, 16.33it/s]

 99%|█████████▉| 143/144 [00:08<00:00, 16.47it/s]

100%|██████████| 144/144 [00:08<00:00, 17.49it/s]

Maior recuperação individual: head 5.5, valor=0.343


In [7]:
candidate_groups = {
    "induction": [(5, 5), (5, 8), (5, 9), (6, 9)],
    "S-inhibition": [(7, 3), (7, 9), (8, 6), (8, 10)],
    "name mover": [(9, 6), (9, 9), (10, 0)],
}

for group, heads in candidate_groups.items():
    print(group)
    for layer, head in heads:
        print(f"  {layer}.{head}: recuperação={head_results[layer, head].item():+.3f}")

induction
  5.5: recuperação=+0.343
  5.8: recuperação=-0.005
  5.9: recuperação=+0.016
  6.9: recuperação=+0.130
S-inhibition
  7.3: recuperação=+0.124
  7.9: recuperação=+0.192
  8.6: recuperação=+0.319
  8.10: recuperação=+0.278
name mover
  9.6: recuperação=-0.048
  9.9: recuperação=+0.264
  10.0: recuperação=+0.084


## Conclusão

Activation patching fornece evidência causal localizada: trocar uma ativação específica altera o comportamento final.

Mas o resultado depende da intervenção escolhida. Fazer patch em uma head inteira ainda não revela **por qual caminho** sua informação chega a outra head.

O próximo notebook isola conexões sender → receiver com path patching e avalia se o circuito proposto é suficiente, completo e mínimo.